In [ ]:
!pip install scikit-learn numpy joblib tkinter pillow

  Using cached pillow-11.3.0-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
Using cached pillow-11.3.0-cp313-cp313-win_amd64.whl (7.0 MB)


In [19]:
import numpy as np
from joblib import load

model = load(r"rforest_model.pkl")
def scale_value(values, mean_value=50, std_value=0.2):
    values = np.array(values, dtype=float)
    scaled_values = (values - mean_value) / std_value
    return scaled_values.tolist()
    
def predict_status(values):
    values = scale_value(values, mean_value=50,std_value=0.2)
    y_pred = model.predict([values])
    if y_pred == 1:
        return "Người này có dấu hiệu của bệnh parkinson"
    else:
        return "Người này không có dấu hiệu của bệnh parkinson"


c:\Users\Admin\Desktop\MHUD\venv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Admin\Desktop\MHUD\venv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk

def add_placeholder(entry, placeholder_text):
    entry.insert(0, placeholder_text)
    entry.bind("<FocusIn>", lambda event: clear_placeholder(entry, placeholder_text))
    entry.bind("<FocusOut>", lambda event: restore_placeholder(entry, placeholder_text))

def clear_placeholder(entry, placeholder_text):
    if entry.get() == placeholder_text:
        entry.delete(0, tk.END)
        entry.config(fg="black")

def restore_placeholder(entry, placeholder_text):
    if not entry.get():
        entry.insert(0, placeholder_text)
        entry.config(fg="gray")

def validate_after(event):
    widget = event.widget
    value = widget.get()
    try:
        float(value)
        widget.config(
            bg="#69f18a",
            highlightcolor="#69f18a",   
            highlightbackground="#69f18a"
        )
    except:
        if value.strip() == "":
            widget.config(bg="white")
        else:
            widget.config(
                bg="#ff8787",
                highlightcolor="#ff8787",
                highlightbackground="#ff8787"
            )

def create_entry_group(parent, title, rows, cols,names):
    label = tk.Label(parent, text=title, font=("Helvetica", 14, "bold"), bg="#f0f2f5")
    label.pack(anchor="w", pady=(30, 5))

    container = tk.Frame(parent, bg="#f0f2f5")
    container.pack()

    for i in range(rows):
        for j in range(cols):
            frame = tk.Frame(container, bg="#f0f2f5")  
            frame.grid(row=i, column=j, padx=8, pady=6)

            entry = tk.Entry(
                frame,
                font=("Helvetica", 10),
                fg="gray",
                width=30,
                relief="solid",         
                bd=1,                   
                highlightthickness=1,   
                bg="white",
                justify="center",
            )
            entry.pack(padx=1, pady=1, ipady=6) 
            entry.bind("<KeyRelease>", validate_after) 
            add_placeholder(entry, f"{names[i][j]}")

            all_entries.append(entry)

def submit_data():
    values = []
    errors = []

    for idx,entry in enumerate(all_entries):
        val = entry.get().strip()

        if val.lower() == f"{namesProperties[idx]}" or val == "":
            errors.append(entry)
            continue

        try:
            num = float(val)
            values.append(num)
        except ValueError:
            errors.append(entry)

    if errors:
        messagebox.showerror("Lỗi nhập liệu", "Vui lòng kiểm tra các ô bị tô đỏ:\n- Không được để trống\n- Chỉ được nhập số")
        for entry in errors:
            entry.config(bg="#ff8787")
        return
    for idx,entry in enumerate(all_entries):
        entry.delete(0, tk.END)
        entry.config(bg="white", fg="gray")
        add_placeholder(entry, f"{namesProperties[idx]}")
    # return values
    predict_status(values)
    messagebox.showinfo("Kết quả", predict_status(values))

    

# Hover cho button
def on_enter(e):
    submit_btn["background"] = "#6B746B"
    submit_btn["foreground"] = "white"

def on_leave(e):
    submit_btn["background"] = "#e7e7e7"
    submit_btn["foreground"] = "black"

root = tk.Tk()
root.title("Chuẩn đoán Parkinson")
root.geometry("1150x720")
root.configure(bg="#f0f2f5")
root.resizable(False, False)
# root.iconbitmap("icon.ico")

main_frame = tk.Frame(root, bg="#f0f2f5")
main_frame.pack(padx=40, pady=20, fill="both", expand=True)

bg_image = Image.open(r"../imgs/background.png")
bg_image = bg_image.resize((1150, 700))
bg_photo = ImageTk.PhotoImage(bg_image)

bg_label = tk.Label(main_frame, image=bg_photo)
bg_label.place(x=0, y=0, relwidth=1, relheight=1)

# Danh sách chứa tất cả entry
all_entries = []
namesProperties = ["MDVP:Fo(Hz)","MDVP:Fhi(Hz)","MDVP:Flo(Hz)","MDVP:Jitter(%)",
                   "MDVP:Jitter(Abs)","MDVP:RAP","MDVP:PPQ","Jitter:DDP",
                   "MDVP:Shimmer","MDVP:Shimmer(dB)","Shimmer:APQ3",
                   "Shimmer:APQ5","MDVP:APQ","Shimmer:DDA",
                   "NHR","HNR","RPDE","DFA","spread1",
                   "spread2","D2","PPE"]
# Tạo các nhóm
create_entry_group(main_frame, "Nhóm tần số cơ bản và dao động tần số", 2, 4, [["MDVP:Fo(Hz)","MDVP:Fhi(Hz)","MDVP:Flo(Hz)","MDVP:Jitter(%)"],["MDVP:Jitter(Abs)","MDVP:RAP","MDVP:PPQ","Jitter:DDP"]])
create_entry_group(main_frame, "Nhóm biên độ và dao động biên độ", 2, 3, [["MDVP:Shimmer","MDVP:Shimmer(dB)","Shimmer:APQ3"],["Shimmer:APQ5","MDVP:APQ","Shimmer:DDA"]])
create_entry_group(main_frame, "Nhóm tỷ lệ tiếng ồn – âm điều hòa", 1, 2, [["NHR","HNR"]])
create_entry_group(main_frame, "Nhóm đặc trưng phi tuyến và hỗn loạn", 2, 3, [["RPDE","DFA","spread1"],["spread2","D2","PPE"]])

submit_btn = tk.Button(
    main_frame,
    text="Chuẩn đoán Parkinson",
    font=("Helvetica", 14, "bold"),
    width=30,
    height=3,
    bg="#e7e7e7",
    relief="groove",
    bd=3,
    cursor="hand2",
    command=submit_data  
)
submit_btn.pack(pady=20)
submit_btn.bind("<Enter>", on_enter)
submit_btn.bind("<Leave>", on_leave)

root.mainloop()
